# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task

The project is a **content-performance prioritisation POC**.

The primary task is **ranking / scoring**: deciding which content pages should be reviewed first.

Three standard ML tasks support the ranking:

1. **Clustering** — group similar pages into performance archetypes.
2. **Classification** — predict whether a page will experience a future decline.
3. **Regression** — predict the size of that decline.
4. **Ranking / scoring** — combine the results to prioritise pages for human review.

Signal analysis and peer-relative comparisons are supporting feature-analysis and feature-engineering steps.

The workflow is:

**observed features → clustering → classification + regression → ranking / scoring**

The final output is a ranked human-review queue.

In [ ]:
import pandas as pd
from pathlib import Path

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Pseudonymized clients: {df['client_id'].nunique():,}")

assert df['content_id'].nunique() == len(df), (
    "Expected one row per pseudonymized content item."
)

ml_tasks = [
    "clustering / archetypes",
    "future-decline classification",
    "future-decline magnitude regression",
    "ranking / scoring",
]
print("ML tasks:", " -> ".join(ml_tasks))


## 2. Target or proxy

### Clustering

Clustering is unsupervised, so it has **no target variable**.

It groups similar pages into archetypes that can also be used as peer groups.

### Classification

The classification target is an observed future outcome such as:

`future_decline_30d`

where:

- `1` = future decline;
- `0` = no future decline.

### Regression

The regression target is the observed size of a future decline, such as:

`future_decline_magnitude_30d`

Classification therefore answers:

**Will the page decline?**

Regression answers:

**How large is the decline likely to be?**

### Ranking / scoring

The final ranking does not use a manually created priority label.

It combines information such as:

**decline probability + expected decline size + page exposure/value**

to rank pages for review.

### Time structure

The final targets must use separate past and future periods:

**past features → decision point → future outcome**

The exact windows and target definitions will be fixed in the data-contract stage.

### Starter-data proxy

The starter dataset is only a trailing-90-day snapshot, so it cannot provide a true future target.

For Assignment 3 only:

`decline_proxy = 1` when `trend_direction == "down"`

This is a temporary framing proxy, not a true future label.

Because `trend_direction` is derived from `trend_pct`, neither field may be used as a predictive feature when this proxy is used.

In [ ]:
# Transparent starter-data proxy for Assignment 3 framing.
required_proxy_columns = {
    "content_id",
    "trend_direction",
    "trend_pct",
    "impressions_prev_30d",
    "impressions_last_30d",
}
missing = sorted(required_proxy_columns.difference(df.columns))
assert not missing, f"Missing required proxy columns: {missing}"

proxy_frame = df[[
    "content_id",
    "impressions_prev_30d",
    "impressions_last_30d",
    "trend_pct",
    "trend_direction",
]].copy()

proxy_frame["decline_proxy"] = (
    proxy_frame["trend_direction"].str.lower().eq("down").astype("int8")
)

print("Starter proxy: decline_proxy = 1 when trend_direction == 'down'.")
print("This is a current-window proxy, not a future causal or intervention label.\n")
print(proxy_frame["decline_proxy"].value_counts().sort_index())
print(f"Proxy positive rate: {proxy_frame['decline_proxy'].mean():.1%}\n")

display(proxy_frame.head(10))

print("Excluded from predictive features for this proxy: ['trend_direction', 'trend_pct']")

# Later warehouse target structure (not fabricated from this snapshot):
target_schema = pd.DataFrame({
    "field": [
        "future_decline_30d",
        "future_decline_magnitude_30d",
    ],
    "role": [
        "future-state classification target",
        "future-decline magnitude regression target",
    ],
    "available_in_starter_snapshot": [False, False],
})
display(target_schema)


## 3. Success metrics

Each ML task has one primary metric and three supporting metrics.

| Task | Primary metric | Secondary metrics |
|---|---|---|
| **Clustering** | **Prediction Strength** | Silhouette Score, Calinski-Harabasz Score, Davies-Bouldin Score |
| **Classification** | **ROC-AUC** | Precision, Recall, F1 Score |
| **Regression** | **RMSE** | MAE, Median Absolute Error, R² |
| **Ranking / scoring** | **Precision@50** | Recall@50, Lift@50, NDCG@50 |

### Clustering

**Prediction Strength** is the primary metric because the archetypes should reproduce on unseen data.

The secondary metrics check cluster separation and compactness.

### Classification

**ROC-AUC** is the primary metric because it measures how well the model separates future declines from non-declines without requiring a fixed classification threshold.

Precision, Recall and F1 are reported to show the practical balance between false alarms and missed declines.

### Regression

**RMSE** is the primary metric because large errors in predicted decline size should be penalised more strongly.

The model must beat a simple baseline that predicts the training-set mean decline magnitude.

MAE, Median Absolute Error and R² provide additional views of prediction error and explained variation.

### Ranking / scoring

**Precision@50** is the primary project metric because the final output is a limited human-review queue.

It measures:

**Of the top 50 recommended pages, how many are genuinely relevant future cases?**

Recall@50 measures coverage, Lift@50 compares the queue with the overall base rate, and NDCG@50 checks whether the most important cases appear near the top.

The learned ranking must be compared with the frozen rule-based baseline using the same pages, future outcomes and `K = 50`.

`K = 50` is the fixed reporting depth for this POC. A real operational review capacity can replace it later if one is provided.

### Supporting analysis

Permutation importance may be used to inspect useful features.

Peer-relative Lift@10% may be used to check whether unusually weak pages within their peer groups are enriched for future declines.

These are supporting analyses, not separate ML tasks.

In [ ]:
# Metric contract for the four standard ML tasks.
import pandas as pd

metric_contract = pd.DataFrame([
    ("Clustering", "Prediction Strength", "Silhouette Score; Calinski-Harabasz Score; Davies-Bouldin Score"),
    ("Classification", "ROC-AUC", "Precision; Recall; F1 Score"),
    ("Regression", "RMSE", "MAE; Median Absolute Error; R²"),
    ("Ranking / scoring", "Precision@50", "Recall@50; Lift@50; NDCG@50"),
], columns=["task", "primary_metric", "secondary_metrics"])

display(metric_contract)
print("Primary project metric: Precision@50")
print("Ranking comparison: learned queue vs frozen rule baseline on the same future outcomes.")
print("K = 50 is the fixed reporting depth for this POC.")


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one pseudonymized content page at a defined decision point**.

Each row should contain only information that would have been available before that decision point, such as:

- search and traffic performance;
- engagement signals;
- content characteristics;
- freshness or update history;
- clustering/archetype information;
- and peer-relative features created from past data.

The starter dataset contains **30,000 rows and 30,000 unique `content_id` values**, so it currently has one row per pseudonymized content page.

For the final warehouse version, the same page may appear at different decision dates. In that case, the unit becomes:

**one content page × one decision point**

The future classification and regression targets belong to the same row but are calculated only from the later outcome window.

The required structure is therefore:

**page at decision point → past features → future decline outcome + future decline magnitude**

Identifiers such as `content_id` and `client_id` may be used for joining, grouping and validation splits, but they are not predictive features.

In [ ]:
# Verify the starter-data grain and show a compact example of one-row-per-page data.
unit_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
]

missing_unit_cols = [c for c in unit_cols if c not in df.columns]
assert not missing_unit_cols, f"Missing unit-of-analysis columns: {missing_unit_cols}"

n_rows = len(df)
n_content = df["content_id"].nunique()

print(f"Rows: {n_rows:,}")
print(f"Unique content_id values: {n_content:,}")
print("Starter unit of analysis: one pseudonymized content page per row")

assert n_rows == n_content, "Starter data is not one row per content page."

print("Identifier roles:")
print("  content_id -> join/group key, not a predictive feature")
print("  client_id  -> grouping/split key, not a predictive feature")

display(df[unit_cols].head())


## 5. Why ML beats a fixed rule here

A fixed rule is useful as a simple baseline, but it is unlikely to capture the full problem.

Content performance depends on several signals at the same time, including visibility, clicks, engagement, freshness, page type and recent performance. The same value can also mean different things for different kinds of pages.

For example, weak CTR may be normal for one type of page but unusual for another. A decline may also matter more when it affects a high-visibility page than a low-visibility page.

The ML approach can combine these signals, learn different page patterns, estimate future decline risk and decline size, and use that information to rank pages.

The fixed rule is still important. It provides a transparent baseline that the ML approach must beat.

If the ML pipeline does not improve the final ranking compared with the simple rule, the simpler rule should be preferred.

In [ ]:
# Simple checks supporting the case for a multivariable ML POC.
supporting_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "content_type",
]

missing_supporting = [c for c in supporting_cols if c not in df.columns]
assert not missing_supporting, f"Missing supporting columns: {missing_supporting}"

print(f"Supporting feature types available: {len(supporting_cols)}")
print(f"Observed content types: {df['content_type'].nunique(dropna=True)}")
print("Baseline rule remains the comparator; ML must improve the final ranking to be kept.")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.